# 04 — Résultats, benchmarking et conclusion

Ce notebook couvre les étapes 7 à 11 de la méthodologie :
7. Résultats
8. Benchmarking : baseline contre modèle fine-tuné
9. Limitations
10. Conclusion
11. Références

**Principe :** les deux modèles sont évalués **dans ce même notebook**, sur **les mêmes questions**, avec **le même chargement** (float16) et **le même décodage** (glouton). Seule différence : l'adaptateur LoRA.

## 0. Installation

In [ ]:
import os, sys

REPO_URL = "https://github.com/eya-bouhmida/finance-text2sql-lora"
REPO_DIR = "/content/finance-text2sql-lora"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    %cd {REPO_DIR}
    !pip install -q transformers accelerate peft datasets
    !pip uninstall -y -q torchao   # conflit avec peft dans Colab, inutile pour ce projet
    sys.path.append(REPO_DIR)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    sys.path.append("..")

DRIVE_DIR = "/content/drive/MyDrive/text2sql-finance-qlora" if IN_COLAB else ".."
DATA_DIR = f"{DRIVE_DIR}/data"
ADAPTER_DIR = f"{DRIVE_DIR}/adapters/qlora-r16"
RESULTS_DIR = f"{DRIVE_DIR}/results"        # sur Drive : les résultats survivent à la session
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
import gc, json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import data as D, model as M, evaluate as E, train as T

pd.set_option("display.max_colwidth", 150)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (active le GPU T4)")
print("Adaptateur trouvé :", os.path.exists(ADAPTER_DIR))

## 1. Jeu de test

In [ ]:
test_records = D.load_jsonl(f"{DATA_DIR}/test.jsonl")
messages = [r["messages"] for r in test_records]
print(f"Questions de test : {len(test_records)}")

## 2. Évaluation des deux modèles

On charge un modèle, on génère, on libère la mémoire GPU, puis on passe au suivant.

In [ ]:
def run_model(adapter_path=None):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model, tokenizer = M.load_model(M.BASE_MODEL, adapter_path=adapter_path)
    outputs, sec = M.generate(model, tokenizer, messages, batch_size=8)
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    del model
    gc.collect(); torch.cuda.empty_cache()
    return outputs, sec, peak_gb

base_out, base_sec, base_mem = run_model(adapter_path=None)
ft_out, ft_sec, ft_mem = run_model(adapter_path=ADAPTER_DIR)

In [ ]:
base_res = E.evaluate_predictions(test_records, base_out)
ft_res = E.evaluate_predictions(test_records, ft_out)

base_sum = E.summarize(base_res)
ft_sum = E.summarize(ft_res)
print("Baseline   :", base_sum)
print("Fine-tuné  :", ft_sum)

## 3. Résultats

Rappel du notebook 02 : la baseline avait obtenu **58,9 %**. Le résultat de la baseline ci-dessus doit être identique (le décodage est déterministe) : c'est une vérification de reproductibilité.

In [ ]:
n = len(test_records)
adapter_mb = T.dir_size_mb(ADAPTER_DIR)

def row(name, s, res, sec, mem, trainable, size):
    k = int(res["execution_match"].sum())
    lo, hi = E.wilson_ci(k, n)
    return {
        "Modèle": name,
        "Execution accuracy (%)": s["execution_accuracy"],
        "IC 95 % (%)": f"[{lo} ; {hi}]",
        "SQL valide (%)": s["valid_sql_rate"],
        "Exact match (%)": s["exact_match"],
        "Temps / question (s)": round(sec, 3),
        "Mémoire GPU inférence (Go)": round(mem, 2),
        "Paramètres entraînés": trainable,
        "Taille à stocker": size,
    }

comparison = pd.DataFrame([
    row("Baseline (Qwen2.5-Coder-1.5B)", base_sum, base_res, base_sec, base_mem, "0", "≈ 3 Go (modèle)"),
    row("Fine-tuné QLoRA (r = 16)", ft_sum, ft_res, ft_sec, ft_mem, "≈ 18,5 M (≈ 1,2 %)", f"{adapter_mb} Mo (adaptateur)"),
]).set_index("Modèle")
comparison.T

## 4. Benchmarking

### 4.1 Le gain est-il significatif ?

Les deux modèles répondent aux **mêmes questions**, on peut donc les comparer question par question. Le **test de McNemar** ne regarde que les questions où ils ne sont pas d'accord :
- questions **corrigées** par le fine-tuning (baseline fausse, fine-tuné juste)
- questions **cassées** par le fine-tuning (baseline juste, fine-tuné faux)

Si la p-value est inférieure à 0,05, l'amélioration n'est pas due au hasard.

In [ ]:
b, c, p_value = E.mcnemar_test(base_res["execution_match"], ft_res["execution_match"])
both_ok = int((base_res["execution_match"] & ft_res["execution_match"]).sum())
both_ko = int((~base_res["execution_match"] & ~ft_res["execution_match"]).sum())

transitions = pd.DataFrame(
    [[both_ok, b], [c, both_ko]],
    index=["Baseline juste", "Baseline fausse"],
    columns=["Fine-tuné juste", "Fine-tuné faux"],
)
display(transitions)
print(f"Questions corrigées par le fine-tuning : {c}")
print(f"Questions cassées par le fine-tuning   : {b}")
print(f"p-value (McNemar exact) : {p_value:.4f} → {'significatif' if p_value < 0.05 else 'NON significatif'} au seuil de 5 %")

### 4.2 Par niveau de complexité

In [ ]:
by_cx = pd.DataFrame({
    "n": base_res.groupby("sql_complexity").size(),
    "Baseline (%)": E.summarize_by(base_res)["execution_accuracy"],
    "Fine-tuné (%)": E.summarize_by(ft_res)["execution_accuracy"],
}).sort_values("n", ascending=False)
by_cx["Gain (points)"] = (by_cx["Fine-tuné (%)"] - by_cx["Baseline (%)"]).round(1)
by_cx

In [ ]:
plot_df = by_cx[["Baseline (%)", "Fine-tuné (%)"]]
labels = [f"{cx}\n(n={k})" for cx, k in zip(by_cx.index, by_cx["n"])]

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(plot_df)); w = 0.38
bars1 = ax.bar(x - w/2, plot_df["Baseline (%)"], w, label="Baseline", color="#8C8C8C")
bars2 = ax.bar(x + w/2, plot_df["Fine-tuné (%)"], w, label="Fine-tuné QLoRA", color="#4C72B0")
ax.bar_label(bars1, fmt="%.0f", padding=2, fontsize=9)
ax.bar_label(bars2, fmt="%.0f", padding=2, fontsize=9)
ax.set_xticks(x, labels); ax.set_ylim(0, 105)
ax.set_ylabel("Execution accuracy (%)")
ax.set_title("Execution accuracy par complexité de requête")
ax.legend(); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/04_accuracy_by_complexity.png", dpi=150)
plt.show()

⚠️ Les catégories avec moins de 10 questions ne permettent pas de conclusion fiable.

### 4.3 Types d'erreurs

In [ ]:
def outcome(res):
    return pd.Series({
        "Correct": int(res["execution_match"].sum()),
        "SQL valide mais faux": int((res["executable"] & ~res["execution_match"]).sum()),
        "SQL invalide": int((~res["executable"]).sum()),
    })

outcomes = pd.DataFrame({"Baseline": outcome(base_res), "Fine-tuné": outcome(ft_res)})
display(outcomes)

ax = (100 * outcomes / n).T.plot.barh(stacked=True, figsize=(9, 2.8),
                                       color=["#55A868", "#DD8452", "#C44E52"])
ax.set_xlabel("% des questions"); ax.set_xlim(0, 100)
ax.set_title("Répartition des résultats"); ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/04_outcomes.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.4 Exemples de questions corrigées et cassées

In [ ]:
merged = base_res[["question", "gold_sql", "pred_sql", "execution_match"]].rename(
    columns={"pred_sql": "baseline_sql", "execution_match": "baseline_ok"})
merged["finetuned_sql"] = ft_res["pred_sql"]
merged["finetuned_ok"] = ft_res["execution_match"]

def show(df, title, k=3):
    print(f"===== {title} ({len(df)}) =====\n")
    for _, r in df.head(k).iterrows():
        print(f"QUESTION  : {r['question']}")
        print(f"RÉFÉRENCE : {r['gold_sql']}")
        print(f"BASELINE  : {r['baseline_sql']}")
        print(f"FINE-TUNÉ : {r['finetuned_sql']}\n" + "-" * 80)

show(merged[~merged["baseline_ok"] & merged["finetuned_ok"]], "Corrigées par le fine-tuning")
show(merged[merged["baseline_ok"] & ~merged["finetuned_ok"]], "Cassées par le fine-tuning")

## 5. Sauvegarde des résultats

In [ ]:
comparison.to_csv(f"{RESULTS_DIR}/04_comparison.csv")
by_cx.to_csv(f"{RESULTS_DIR}/04_by_complexity.csv")
merged.to_csv(f"{RESULTS_DIR}/04_predictions_side_by_side.csv", index=False)
E.save_json({
    "n_test": n,
    "baseline": {**base_sum, "sec_per_question": round(base_sec, 3)},
    "finetuned": {**ft_sum, "sec_per_question": round(ft_sec, 3)},
    "mcnemar": {"fixed": c, "broken": b, "p_value": round(p_value, 5)},
    "adapter_mb": adapter_mb,
}, f"{RESULTS_DIR}/04_final_metrics.json")
print("Résultats sauvegardés sur Drive :", RESULTS_DIR)
print("→ Télécharge ce dossier et copie-le dans results/ du repo pour le versionner sur GitHub.")

## 6. Synthèse chiffrée (générée automatiquement)

In [ ]:
gain = ft_sum["execution_accuracy"] - base_sum["execution_accuracy"]
print(f"""
Sur {n} questions financières, le fine-tuning QLoRA fait passer l'execution accuracy
de {base_sum['execution_accuracy']} % à {ft_sum['execution_accuracy']} % ({gain:+.1f} points).
Le taux de SQL valide passe de {base_sum['valid_sql_rate']} % à {ft_sum['valid_sql_rate']} %.
Le fine-tuning corrige {c} questions et en casse {b} (McNemar p = {p_value:.4f}).
L'adaptateur pèse {adapter_mb} Mo et a été entraîné en ≈ 18,5 minutes sur un GPU T4 gratuit.
""")

## 7. Limitations

- **Données synthétiques** : le dataset Gretel est généré par un LLM. Les schémas sont petits (1 à 3 tables) et propres, loin des bases bancaires réelles (centaines de tables, noms de colonnes cryptiques, valeurs sales).
- **Jeu de test réduit** : 158 questions, soit une marge d'incertitude d'environ ±8 points sur l'execution accuracy (voir les intervalles de confiance). Les catégories `window functions` et `subqueries` comptent moins de 10 questions.
- **Même distribution** : le train et le test viennent du même générateur. Le gain mesuré peut en partie refléter l'adaptation au *style* de Gretel plutôt qu'une meilleure compréhension du SQL en général.
- **Métrique** : l'execution accuracy compare des ensembles de lignes. Une réponse correcte avec une colonne en plus est comptée fausse, et une requête fausse peut tomber juste par hasard sur une petite base.
- **Une seule configuration** : un seul rang (r = 16), une seule graine aléatoire, pas d'étude d'ablation.
- **Précision** : l'adaptateur a été entraîné sur le modèle en 4 bits, puis appliqué au modèle en float16 pour l'évaluation (procédure courante, mais légère différence entre entraînement et inférence).
- **SQLite et anglais uniquement.**

**Perspectives :** évaluer sur la base bancaire réelle `financial` du benchmark BIRD, comparer plusieurs rangs LoRA (4, 8, 16, 32), comparer avec un grand modèle propriétaire, et tester des questions en français.

## 8. Conclusion

Ce projet montre qu'un petit modèle open source (Qwen2.5-Coder, 1,5 milliard de paramètres), fine-tuné avec QLoRA sur un seul GPU gratuit, améliore nettement la génération de requêtes SQL sur des bases financières : l'execution accuracy passe de **58,9 %** à **71,5 %** (**+12,7 points**) sur 158 questions de test. Ce gain est **statistiquement significatif** (test de McNemar, p = 0,0002) : le fine-tuning corrige 24 questions et n'en casse que 4. Le modèle ne produit presque plus de SQL invalide (de 12 à 2 requêtes), et progresse surtout sur les agrégations (+19 points). Ce résultat a été obtenu en n'entraînant qu'environ **1,2 % des paramètres**, en **18,5 minutes** sur un GPU T4 gratuit, avec un adaptateur de **85 Mo**, et le modèle reste déployable en interne, sans envoyer de données à une API externe. Le principal point faible reste la logique des requêtes : 43 requêtes s'exécutent mais renvoient un mauvais résultat. Ces résultats restent à confirmer sur des bases bancaires réelles, plus complexes que les données synthétiques utilisées ici.

## 9. Références

1. Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR 2022. https://arxiv.org/abs/2106.09685
2. Dettmers et al., *QLoRA: Efficient Finetuning of Quantized LLMs*, NeurIPS 2023. https://arxiv.org/abs/2305.14314
3. Li et al., *Can LLM Already Serve as a Database Interface? (BIRD)*, NeurIPS 2023. https://arxiv.org/abs/2305.03111
4. Kumar et al., *BookSQL: A Large Scale Text-to-SQL Dataset for Accounting Domain*, NAACL 2024. https://aclanthology.org/2024.naacl-long.28
5. Zhang et al., *FinSQL: Model-Agnostic LLMs-based Text-to-SQL Framework for Financial Analysis*, SIGMOD 2024. https://arxiv.org/abs/2401.10506
6. Gretel AI, *synthetic_text_to_sql* (dataset, Apache 2.0). https://huggingface.co/datasets/gretelai/synthetic_text_to_sql
7. Qwen Team, *Qwen2.5-Coder-1.5B-Instruct* (modèle, Apache 2.0). https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct